In [2]:
import numpy as np
import sunpy.map
import os
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import warnings
from queue import Queue
import re
from datetime import datetime
from astropy.io import fits
import astropy.units as u
from astropy.coordinates import SkyCoord
%matplotlib notebook

In [3]:
file_path = "E:/python/projects/alfven/data/corrected/"
output_path = "E:/python/projects/alfven/data/corrected/"

In [4]:
files = [os.path.abspath(os.path.join(file_path, file)) for file in os.listdir(file_path) if file.endswith('.fits')]
def extract_datetime(file_name):
    match = re.search(r'\d{8}T\d{6}', file_name)
    if match:
        return datetime.strptime(match.group(), '%Y%m%dT%H%M%S')
    else:
        return datetime.min

files = sorted(files, key=extract_datetime)

In [5]:
def transform(file, save_path):
    old_map = sunpy.map.Map(file)
    
    origin = SkyCoord(-100*u.arcsec, -2800*u.arcsec, frame=old_map.coordinate_frame)
    out_shape = (2500, 2500)
    out_header = sunpy.map.make_fitswcs_header(
        out_shape,
        origin,
        scale=[0.492, 0.492]*u.arcsec/u.pix,
        projection_code="TAN"
    )
    supplement_list = [
        'DSUN_OBS',
        'RSUN_OBS',
        'RSUN_ARC',
        'FILENAME'
    ]
    unit_trasnform = {
        'CUNIT1': 'arcsec',
        'CUNIT2': 'arcsec',
        'CDELT1': 0.492,
        'CDELT2': 0.492,
        'CRVAL1': -100,
        'CRVAL2': -2800
        }
    
    out_map = old_map.reproject_to(out_header)
    for key in supplement_list:
        out_map.meta[key] = old_map.meta[key]
    for key, value in unit_trasnform.items():
        out_map.meta[key] = value
    output_name = os.path.join(save_path, os.path.basename(file))
    out_map.save(output_name, filetype='fits', overwrite=True)

In [6]:
with ThreadPoolExecutor() as executor:
    futures = {executor.submit(transform, file, output_path+'transform_data/'): file for file in files}

    for future in tqdm(as_completed(futures), total=len(files), desc="Processing files"):
        future.result()

Processing files:   0%|          | 0/600 [00:00<?, ?it/s]

In [7]:
transform_files = [os.path.abspath(os.path.join("./data/corrected/transform_data/", file))
                   for file in os.listdir("./data/corrected/transform_data/")]
transform_files = sorted(transform_files, key=extract_datetime)

In [8]:
window_size = 3
mid = window_size // 2
q_data = Queue()
head = 0
sum_data = np.zeros_like(sunpy.map.Map(transform_files[0]).data)
save_path = "./data/corrected/3moving/"

for point in tqdm(range(len(transform_files)+1), desc='moving average', total=len(files)+1):
    if q_data.qsize() == window_size:  # 队列中有window_size个元素，进行计算平均值操作
        name = os.path.basename(transform_files[head+mid])  # 得到保存的文件名
        mean_data = sum_data / window_size # 平均数据
        _, header = fits.getdata(transform_files[head+mid], header=True) # 获取头信息
        mean_map = sunpy.map.Map(mean_data, header)
        mean_map.save(save_path + name, filetype='fits', overwrite=True) # 制作并储存平均后的文件
        
        if point<len(transform_files):
            sum_data -= q_data.get()  # 出队并从总数据减去队头数据
            data_in = fits.getdata(transform_files[point])
            sum_data += data_in
            q_data.put(data_in)
            head += 1

    elif q_data.qsize() < window_size:
        data_in = fits.getdata(transform_files[point])
        q_data.put(data_in)
        sum_data += data_in  # 增加初始化时的数据累加
         
    else:
        raise RuntimeError("Something wrong in the algorithm.")

moving average:   0%|          | 0/601 [00:00<?, ?it/s]